In [ ]:
import sys

import sbdynt as sbd

# Default Proper Element Runs for TNOs and Asteroids

Default properties for computing proper elements for TNOs is contained within the same ``run_tno`` function used to initialize and run the machine learning algorithm for TNO resonance occupation.
A similar function, ``run_asteroid`` can be used to similarly run and compute proper elements for asteroids. We demonstrate these functions below for TNO (15760) Albion and the dwarf planet and asteroid (1) Ceres. 

To compute proper elements as well, from the ``run_tno`` function, user should set the boolean variable ``run_proper = True``, which will compute the proper elements and save the results to the ``tno_result`` output. We reccomend also setting ``run_ML=True`` to save time in the event the TNO is a scattering object (if the most common classification is scattering, then by default, the 150 Myr proper element integration is not run)

The bulk of the time required by these functions is actually spent integrating the simulation. The actual computation of the proper elements is comparatively fast. Proper elements can be re-calculated from an existing simulation archive file using the ``analyze_tno_run`` or ``analyze_ast_run`` functions.

In this TNO example, we will still run the machine learning algorithm by including ``run_ML = True``, as this produces a more complex Simulation Archive with varying time resolution. This demonstrates how these complex archives are automatically handled while computing the proper elements from the simulation. We also use ``logfile='screen'``, which will print timestamps and intermediate results to demonstrate and benchmark how long the default runs may take to fully integrate. (Setting ``logfile=<other-string>`` will most of information to a file with that name, minus the printed results, which can be accessed from the returned ``tno_result`` and ``ast_result`` functions.)

See Spencer et al. (2026) for details on how proper elements are calculated

## TNO example -- 150 Myr proper element integration

**NOTE OF CAUTION:** we know we have chosen a classical belt TNO as an example, so we are setting ``integrator='whfast'`` to speed up the integration so this example only takes ~5 minutes to fully execute. **The safest thing to do is to not pass ``integrator`` to ``run_tno``** as that will result in the use of the mercurius hybrid integrator, which can correctly handle a TNO's close encounters with Neptune. If ``run_ML=True`` is set, that short integration will be done with mercurius regardless of the value of ``integrator``, so you will have some warning if the object is actively scattering. But currently stable TNOs can scatter in the future, so use ``integrator='whfast'`` with caution!

The function ``run_tno`` returns a flag (0 if it failed, 1 if it succeeded), the ``tno_result`` class, and the rebound simulation instance ``sim``, which is a snapshot at +75 Myr (the 150 Myr integration is split half backwards and half forwards by default)


In [ ]:
flag, tno_result, sim = sbd.run_tno(
    des="Albion",
    clones=None,
    datadir="outputs-from-example-notebooks",
    archivefile=None,
    logfile="screen",
    deletefile=True,
    run_ML=True,
    run_proper=True,
    integrator="whfast",
)

**If you already ran an integration and have the archive file saved, you can use ``analyze_tno_run`` to quickly re-calculate the proper elements**

In [ ]:
flag, tno_result = sbd.analyze_tno_run(
    des="Albion",
    clones=None,
    datadir="outputs-from-example-notebooks",
    archivefile=None,
    logfile="screen",
    run_ML=False,
    run_proper=True,
)

## Asteroid example -- 10 Myr proper element integration

**NOTE OF CAUTION:** we again set ``integrator='whfast'`` to speed up the integration so this example only takes ~5 minutes to fully execute. Use ``integrator='whfast'`` with caution!

Note that here we set ``clones=0`` to run only the best-fit orbit. Setting ``clones=None`` will do the same 3-sigma clones as in the TNO example abobe, and setting it to some value >0 will produce the specified number of clones sampled in a Guassian manner from the object's covariance matrix in JPL's small body database.

The outputs are the same as in the TNO example above

In [ ]:
flag, ast_result, sim = sbd.run_ast(
    des="Ceres",
    clones=None,
    datadir="outputs-from-example-notebooks",
    logfile="screen",
    deletefile=True,
    run_stability=False,
    integrator="whfast",
)

In [ ]:
flag, ast_result = sbd.analyze_ast_run(
    des="Ceres", clones=None, datadir="outputs-from-example-notebooks", logfile="screen", run_stability=False
)

## Getting more information from ``tno_result`` and ``ast_result``

We can see the results for the best-fit orbit in an easy way by calling the ``print_results`` function within the proper_element class object inside of the result.

In [ ]:
tno_result.proper_elements.print_results()
ast_result.proper_elements.print_results()

# Proper Element Outputs

The proper element results are saved to a ``proper_element`` class object within the ``tno_result`` object. 
This class object contains a large number of relevant values and indicators related to the orbital evolution of the small body corresponding to the proper motion over time. These include...

```proper_elements```

```mean_elements```

```osculating_elements```

```proper_errors```

```planet_freqs```

```proper_windows```

### Proper Elements

The proper elements themselves are contained in a dictionary with the same name as the ``proper_element`` class. These can be contrasted with the mean elements, (which are simply the mean of the osculating element time array), and the initial osculating elements, (which represent the orbital elements at time ``t=0`` in the simulation).

We note that these mean elements are not be confused with the ``mean`` indicator, which is used to identify objects which experience chaotic motion or long-term periodic evolution such that the synethic proper element cannot be accurately computed for the small-body in the given integration. 
We will discuss the ``mean`` indicator, as well as other useful indicators, in more detail in the ``proper_elements_advanced`` notebook. 

These three outputs contain the semi-major axis, eccentricity, inclination, as well as the proper and mean precession rates, ``g`` and ``s``, in the case of the ``proper_elements`` and ``mean_elements`` variables. The ``osculating_elements`` variable instead reports the initial ``omega`` and ``Omega`` values at the ``t=0`` epoch.

The proper and mean ``g`` and ``s`` frequencies are reported in both revolutions/year (the units of measurement used by the filtering, the direct result from the filter) and arcseconds/year (the typically reported value for the proper precession rates).

In [ ]:
print("Albion: ", tno_result.proper_elements.proper_elements)
print("\nCeres:", ast_result.proper_elements.proper_elements)

In [ ]:
print("Albion:", tno_result.proper_elements.mean_elements)
print("\nCeres:", ast_result.proper_elements.mean_elements)

In [ ]:
print("Albion:", tno_result.proper_elements.osculating_elements)
print("\nCeres:", ast_result.proper_elements.osculating_elements)

### Proper Element Uncertainties

The proper element uncertainties are contained in the ``proper_errors`` dictionary.

In [ ]:
print("Albion:", tno_result.proper_elements.proper_errors)
print("\nCeres:", ast_result.proper_elements.proper_errors)

### Planetary Frequencies

Users interested in the identified planetary secular frequencies can retrieve these from the ``planets`` dictionary. These are only reported in the units of rev/yr. Users can retrieve the "/yr units by multipying these results by 1296000.

In [ ]:
print("Albion:", tno_result.proper_elements.planet_freqs)
print("\nCeres:", ast_result.proper_elements.planet_freqs)

### Window Proper Elements

Users may also access the proper element computed for each of the windows. This could be useful in cases of objects which experience chaotic or scattering motion, if the users wishes to see the proper element before such events. 

In [ ]:
print("Albion:", tno_result.proper_elements.proper_windows)
print("\nCeres:", ast_result.proper_elements.proper_windows)

More advanced outputs are further discussed in the ``proper_elements_advanced.ipynb`` file, which include flags related to chaos, the amplitude of secular terms, as well as some indicators which may be hekpful for identifying secualr resonance occupation. 